# [Baseline_Train] — 모델 실험판

KBO 투구 하나가 **제구 성공** 투구일 확률을 예측합니다.

기존 baseline과 다른 점: `config["model"]` 하나만 바꾸면 RandomForest, LightGBM, ExtraTrees, XGBoost, CatBoost, 로지스틱회귀 중 골라서 실험할 수 있습니다. 검증(5)·재학습&저장(6)·제출파일 생성(7)은 787.04점을 냈던 로직 그대로입니다.

## 1. 라이브러리 불러오기

In [1]:
!pip install lightgbm xgboost catboost -q

In [2]:
import os
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## 2. 변수 설정

경로와 데이터의 기본 설정을 여기서 한곳에 모아 관리합니다. `DATA_DIR`만 본인 환경에 맞게 수정하면 나머지는 자동으로 따라갑니다.

In [ ]:
DATA_DIR = r"C:\Users\user\Desktop\data" 
MODEL_DIR = os.path.join(DATA_DIR, "model")

ID = "row_id"
TARGET = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]

print("DATA_DIR:", DATA_DIR)
print("TARGET:", TARGET)
print("CAT_COLS:", CAT_COLS)

DATA_DIR: C:\Users\user\Desktop\data
TARGET: control_success
CAT_COLS: ['top_bottom', 'game_type', 'base_state']


## 3. 실험 설정 — 모델 선택

`config["model"]`만 바꿔서 아래 셀들을 다시 실행하면 다른 모델로 바로 실험할 수 있습니다.

In [4]:
config = {
    "model": "lgb",   # "rf", "et", "logistic", "lgb", "xgb", "cbt" 중 선택
}

params = {
    "logistic": {
        "n_jobs": -1,
        "random_state": 42,
        "max_iter": 300,
        "penalty": "l2",
    },
    "et": {
        "n_jobs": -1,
        "random_state": 42,
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 200,
    },
    "rf": {
        "n_jobs": -1,
        "random_state": 42,
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 200,
    },
    "lgb": {
        "random_state": 42,
        "objective": "binary",
        "n_jobs": -1,
        "verbosity": -1,
        "n_estimators": 233,        # early stopping으로 찾은 최적 트리 개수 (Val 680.08)
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": 6,
        "min_child_samples": 300,
        "reg_alpha": 1.0,
        "reg_lambda": 2.0,
        "subsample": 0.7,
        "colsample_bytree": 0.6,
    },
    "xgb": {
        "random_state": 42,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_jobs": -1,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "max_depth": 6,
        "reg_lambda": 1,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "tree_method": "hist",
    },
    "cbt": {
        "random_seed": 42,
        "objective": "Logloss",
        "verbose": 0,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "max_depth": 6,
        "l2_leaf_reg": 3,
        "subsample": 0.7,
        "task_type": "CPU",
        "allow_writing_files": False,
    },
}


def build_model(config, params):
    m = config["model"]
    model_params = params[m]
    if m == "logistic":
        return LogisticRegression(**model_params)
    elif m == "et":
        return ExtraTreesClassifier(**model_params)
    elif m == "rf":
        return RandomForestClassifier(**model_params)
    elif m == "lgb":
        return LGBMClassifier(**model_params)
    elif m == "xgb":
        return XGBClassifier(**model_params)
    elif m == "cbt":
        return CatBoostClassifier(**model_params)
    else:
        raise ValueError(f"알 수 없는 model: {m}")


print("현재 실험 모델:", config["model"])

현재 실험 모델: lgb


## 4. 데이터 불러오기

`train.csv` 는 2019~2024 시즌이고 평가 데이터는 2025 시즌입니다.

사용할 피처 목록은 `test.csv` 가 정합니다. 여기에 팀 baseline에서 검증된 피처엔지니어링(14개 파생 피처 추가, 중복 2개 제거)을 그대로 적용합니다.

In [5]:

removal_features = {
    ID,   # 식별자, 피처로 쓰면 안 됨
     "asof_pitcher_prev5_game_success_rate",
     "asof_pitcher_strike_rate"            

}

In [6]:
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns

# removal_features 에 있는 컬럼만 뺀 상태 (파생 피처는 아직 없음)
FEATURES = [c for c in test_cols if c not in removal_features]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig",
                    usecols=FEATURES + [TARGET])

print("train:", train.shape, "| 피처:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")

train: (1475092, 46) | 피처: 45 (범주형 3, 수치형 42)
시즌: 2019 ~ 2024
제구 성공률: 0.5238


## 5. 전처리 정의

범주형 3개(`top_bottom`, `game_type`, `base_state`)는 정수로 바꾸고, 수치형 피처의 결측값은 중앙값으로 채웁니다. 모델 종류와 무관하게 이 전처리는 그대로 사용합니다.

In [7]:
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                           unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])

## 6. 모델 정의와 학습

`config["model"]`에 지정된 모델로 학습합니다. 검증 분할(2024시즌 홀드아웃)은 팀 baseline과 동일합니다. 다른 모델을 시험해보려면 1.5번 셀에서 `config["model"]`을 바꾸고, 이 셀부터 다시 실행하세요.

In [8]:
model = Pipeline([
    ("pre", preprocessor),
    ("clf", build_model(config, params)),
])

# 2024 시즌을 검증용으로 떼어 두고 2019~2023 으로 학습합니다.
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET]
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET]
print("train:", len(X_train), "| val:", len(X_val))

t = time.time()
model.fit(X_train, y_train)
print(f"학습 완료 :: {time.time() - t:.1f}s")

train: 1221585 | val: 253507
학습 완료 :: 10.3s


## 7. 검증 — Brier Skill Score

학습 데이터에서 떼어 둔 2024 시즌으로 검증 점수를 계산합니다.

Brier 는 예측 확률과 실제값(0/1) 차이의 제곱 평균이고, 이를 상수 예측의 Brier 인 `r(1-r)` 로 나누어 Brier Skill Score 를 구합니다.

In [9]:
val_pred = model.predict_proba(X_val)[:, 1]

r = y_val.mean()
brier = ((val_pred - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"모델: {config['model']}")
print(f"Brier: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"Validation Score: {score:.2f}")

모델: lgb
Brier: 0.248346 | 기준선 r(1-r): 0.249807
Validation Score: 584.95


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 8. 전체 데이터로 재학습 & 모델 저장

검증으로 성능을 확인했으니 이제 전체 학습 데이터로 다시 학습합니다.

학습한 파이프라인을 `./model/rf.pkl` 로 저장합니다. (파일명은 모델 종류와 무관하게 `rf.pkl`로 고정 — `script.py`가 이 이름을 참조하므로 변경하면 안 됩니다.) 이 파일을 추론용 `script.py`, `requirements.txt` 와 함께 `baseline_submit.zip` 으로 묶으면 제출 준비가 끝납니다.

In [10]:
t = time.time()
model.fit(train[FEATURES], train[TARGET])
print(f"재학습 완료 :: {time.time() - t:.1f}s")

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, os.path.join(MODEL_DIR, "rf.pkl"), compress=3)
print("저장 완료:", os.path.join(MODEL_DIR, "rf.pkl"))

재학습 완료 :: 12.3s
저장 완료: C:\Users\user\Desktop\data\model\rf.pkl


## 9. 제출 파일 자동 생성 — `baseline_submit.zip`

대회 평가 가이드가 요구하는 zip 구조는 다음과 같습니다 (최상위에 이 3개만 있어야 함):

```
baseline_submit.zip
├── model/
│      └── rf.pkl
├── script.py
└── requirements.txt
```

`data/`, `output/`은 평가 서버가 제출 시 자동으로 추가하는 디렉토리이므로 우리가 만들면 안 됩니다. 추가 최상위 폴더로 감싸져 있으면 설치 오류가 나므로, zip을 만든 뒤 실제 구조를 다시 열어서 검증까지 자동으로 수행합니다.

`requirements.txt`는 지금 이 노트북(커널)의 라이브러리 버전을 그대로 읽어와 자동 생성합니다 — `rf.pkl`을 학습한 환경과 항상 일치해야 평가 서버(Python 3.11.15)에서 모델을 문제없이 불러올 수 있습니다.

`script.py`(`DATA_DIR`에 위치)에도 위 2번 셀의 `engineer_features()`가 **동일하게** 들어있는지 확인하세요. 어긋나면 채점 서버에서 컬럼 불일치로 실패합니다.

In [11]:
import shutil
import zipfile
import sklearn
import lightgbm
import xgboost
import catboost

# ---- 경로 설정 ----
SUBMIT_DIR = os.path.join(DATA_DIR, "submit_build")      # 임시 작업 폴더
SUBMIT_SCRIPT_SRC = os.path.join(DATA_DIR, "script.py")  # 대회 제공 script.py 위치
ZIP_PATH = os.path.join(DATA_DIR, "baseline_submit")     # .zip 은 자동으로 붙음

# 평가 서버가 자동으로 채워주는 디렉토리 — 우리가 만들면 안 됨
FORBIDDEN_TOP_LEVEL = {"data", "data/", "output", "output/"}
REQUIRED_TOP_LEVEL = {"model/", "script.py", "requirements.txt"}

# ---- 0) 작업 폴더 초기화 (이전 빌드 잔여물 방지) ----
if os.path.exists(SUBMIT_DIR):
    shutil.rmtree(SUBMIT_DIR)
os.makedirs(os.path.join(SUBMIT_DIR, "model"), exist_ok=True)

# ---- 1) 현재 커널의 정확한 라이브러리 버전으로 requirements.txt 생성 ----
# 사용한 모델에 필요한 라이브러리만 최소로 담습니다 (설치 시간 10분 제한 고려).
req_lines = [
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={joblib.__version__}",
    f"pandas=={pd.__version__}",
]
if config["model"] == "lgb":
    req_lines.append(f"lightgbm=={lightgbm.__version__}")
elif config["model"] == "xgb":
    req_lines.append(f"xgboost=={xgboost.__version__}")
elif config["model"] == "cbt":
    req_lines.append(f"catboost=={catboost.__version__}")

print("현재 환경 버전 (requirements.txt에 반영):")
for line in req_lines:
    print(" ", line)

with open(os.path.join(SUBMIT_DIR, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(req_lines) + "\n")

# ---- 2) rf.pkl, script.py 를 작업 폴더로 복사 ----
shutil.copy(os.path.join(MODEL_DIR, "rf.pkl"), os.path.join(SUBMIT_DIR, "model", "rf.pkl"))

if not os.path.exists(SUBMIT_SCRIPT_SRC):
    raise FileNotFoundError(
        f"script.py 를 찾을 수 없음: {SUBMIT_SCRIPT_SRC}\n"
        "대회에서 받은 script.py 를 이 경로에 두거나 SUBMIT_SCRIPT_SRC 를 수정하세요."
    )
shutil.copy(SUBMIT_SCRIPT_SRC, os.path.join(SUBMIT_DIR, "script.py"))

# ---- 3) zip 묶기 (SUBMIT_DIR '내용물'을 최상위에 바로 압축 — 폴더로 감싸지 않음) ----
if os.path.exists(f"{ZIP_PATH}.zip"):
    os.remove(f"{ZIP_PATH}.zip")
shutil.make_archive(ZIP_PATH, "zip", SUBMIT_DIR)

# ---- 4) 구조 검증: 최상위가 정확히 model/, script.py, requirements.txt 인지 확인 ----
with zipfile.ZipFile(f"{ZIP_PATH}.zip") as zf:
    names = zf.namelist()
    top_level = {n.split("/")[0] + ("/" if "/" in n else "") for n in names}

    if top_level & FORBIDDEN_TOP_LEVEL:
        raise RuntimeError(
            f"data/ 또는 output/ 이 zip에 포함되어 있음 (평가 서버가 자동 추가하는 "
            f"디렉토리라 포함하면 안 됨): {top_level}"
        )
    if top_level != REQUIRED_TOP_LEVEL:
        raise RuntimeError(
            f"zip 최상위 구조가 규격과 다름.\n"
            f"  기대: {REQUIRED_TOP_LEVEL}\n"
            f"  실제: {top_level}\n"
            "추가 최상위 폴더로 감싸져 있지 않은지 확인하세요."
        )
    print("✅ 구조 검증 통과 — model/, script.py, requirements.txt 만 최상위에 존재")
    print("포함 파일:")
    for name in names:
        print(" -", name)

print(f"\n✅ 생성 완료: {ZIP_PATH}.zip")

현재 환경 버전 (requirements.txt에 반영):
  scikit-learn==1.7.2
  joblib==1.5.2
  pandas==2.3.3
  lightgbm==4.6.0
✅ 구조 검증 통과 — model/, script.py, requirements.txt 만 최상위에 존재
포함 파일:
 - model/
 - requirements.txt
 - script.py
 - model/rf.pkl

✅ 생성 완료: C:\Users\user\Desktop\data\baseline_submit.zip


## 10. 여러 모델 한 번에 비교해보기 (선택)

1.5번 셀의 `config["model"]`을 하나씩 바꿔가며 4~5번을 반복 실행하는 대신, 아래처럼 한 번에 돌려서 결과를 표로 모아볼 수도 있습니다. (제출용 `rf.pkl`은 덮어쓰지 않고, 비교만 합니다.)

In [12]:
results = []

for model_name in ["rf", "lgb", "xgb", "cbt"]:
    config["model"] = model_name
    trial_model = Pipeline([("pre", preprocessor), ("clf", build_model(config, params))])

    t = time.time()
    trial_model.fit(X_train, y_train)
    fit_time = time.time() - t

    val_pred = trial_model.predict_proba(X_val)[:, 1]
    brier = ((val_pred - y_val) ** 2).mean()
    score = max(0, 100000 * (1 - brier / baseline_brier))

    print(f"{model_name}: Validation Score {score:.2f} (학습 {fit_time:.1f}s)")
    results.append({"model": model_name, "score": score, "fit_time": fit_time})

results_df = pd.DataFrame(results).sort_values("score", ascending=False)
print("\n=== 비교 결과 ===")
print(results_df)

rf: Validation Score 431.63 (학습 29.1s)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgb: Validation Score 584.95 (학습 10.8s)
xgb: Validation Score 443.91 (학습 13.4s)
cbt: Validation Score 656.16 (학습 20.9s)

=== 비교 결과 ===
  model       score   fit_time
3   cbt  656.159239  20.875125
1   lgb  584.952184  10.756587
2   xgb  443.913767  13.449745
0    rf  431.629706  29.144634
